<a href="https://colab.research.google.com/github/NirtonAfonso/tech-challenge-fase3-medflow-ai/blob/develop/notebooks/02_fine_tuning_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 02 — Fine-tuning QLoRA da LLM clínica

**Tech Challenge Fase 3 — MedFlow AI**

| | |
|---|---|
| Runtime | ⚠️ **GPU obrigatória** — `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU** (T4 do nível gratuito basta) |
| Como rodar | selecione a GPU e use `Executar tudo` |
| Saída no Drive | `MedFlowAI_Fase3/02_fine_tuning/` (`adapter/`, `artifacts/`, `checkpoints/`, `logs/`, `bundles/`) |
| Duração estimada | 20–60 min em T4, conforme fila do Colab |

## O que este notebook faz

1. diagnostica GPU e **escolhe a precisão pelo hardware** (T4 → FP16; Ampere+ → BF16);
2. clona a branch `develop` e instala tudo;
3. constrói o dataset SFT (independente do banco);
4. mede o **baseline do modelo base antes do treino**;
5. treina QLoRA de verdade;
6. salva o adapter no Google Drive e **prova que a recarga funciona**;
7. compara `base` × `fine_tuned` × `fine_tuned + RAG` no mesmo split;
8. gera `medflow_colab_results.zip` (só métricas) e valida o pacote.

> **Nada aqui é fabricado.** Sem GPU, o pipeline recusa treinar e nenhum arquivo de métrica é escrito.
> O ZIP final é o que você devolve para atualizar README e relatório.

In [ ]:
# @title ▶ Bootstrap — execute esta célula primeiro (Colab ou local)
#
# Prepara tudo do zero em um runtime Colab novo: monta o Google Drive, clona a
# branch `develop`, instala as dependências e cria a estrutura de saída.
# Rodando localmente, detecta o repositório e pula clone/Drive.

import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/NirtonAfonso/tech-challenge-fase3-medflow-ai.git"
REPO_BRANCH = "develop"
REPO_DIR = "tech-challenge-fase3-medflow-ai"
NOTEBOOK_ID = "02_fine_tuning"
REQUIREMENTS = "requirements-colab.txt"

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))


def _run(*args, **kwargs):
    return subprocess.run(list(args), check=kwargs.pop("check", True), **kwargs)


def _pip(*args):
    _run(sys.executable, "-m", "pip", *args)


def _tem_torch_cuda() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except Exception:
        return False


# --- 1. Google Drive ---------------------------------------------------------
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        print("Google Drive montado em /content/drive")
    except Exception as erro:
        print(f"ATENÇÃO: falha ao montar o Drive ({erro}).")
        print("Os resultados ficarão apenas em /content e serão PERDIDOS ao encerrar a sessão.")

# --- 2. Repositório ----------------------------------------------------------
def _raiz_local() -> pathlib.Path | None:
    atual = pathlib.Path.cwd()
    for candidato in [atual, *atual.parents]:
        if (candidato / "src" / "medflow_ai").exists():
            return candidato
    return None


raiz = _raiz_local()
if raiz is None:
    destino = pathlib.Path("/content" if IN_COLAB else ".") / REPO_DIR
    if destino.exists():
        _run("git", "-C", str(destino), "fetch", "--depth", "1", "origin", REPO_BRANCH)
        _run("git", "-C", str(destino), "checkout", REPO_BRANCH)
        _run("git", "-C", str(destino), "pull", "--ff-only", "origin", REPO_BRANCH)
    else:
        # Sempre com --branch explícita: nunca clonar a default implicitamente.
        _run("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(destino))
    raiz = destino.resolve()

os.chdir(raiz)
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
print(f"Raiz do projeto: {raiz}")

# --- 3. Dependências ---------------------------------------------------------
_pip("install", "-q", "-U", "pip")
_pip("install", "-q", "-r", REQUIREMENTS)
_pip("install", "-q", "-e", ".")

# Dependências de treinamento. O Colab já traz torch com CUDA compatível:
# reinstalar quebraria a build e demoraria minutos à toa.
_torch_ok = _tem_torch_cuda()
print("torch com CUDA já disponível:", _torch_ok)
_pacotes = [
    "transformers>=4.51", "peft>=0.14", "trl>=0.15", "datasets>=3.0",
    "accelerate>=1.0", "bitsandbytes>=0.45", "sentencepiece",
]
if not _torch_ok:
    _pacotes.insert(0, "torch")
_pip("install", "-q", "-U", *_pacotes)

# --- 4. Diagnóstico ----------------------------------------------------------
import platform

from medflow_ai.colab import ensure_structure, git_info, in_colab, write_run_metadata

_git = git_info(raiz)
print("\n" + "=" * 78)
print(f"Python        : {platform.python_version()}")
print(f"Ambiente      : {'Google Colab' if IN_COLAB else 'local'}")
print(f"Branch        : {_git['branch']}")
print(f"Commit        : {_git['commit']}")
print("=" * 78)

# --- 5. GPU ------------------------------------------------------------------
from medflow_ai.fine_tuning.precision import resolve_precision
_politica = resolve_precision()
print(_politica.explain())
if not _politica.trainable:
    print("\n>>> ATENÇÃO: este notebook exige GPU.")
    print(">>> Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4 basta).")

# --- 6. Estrutura de saída (Drive no Colab, artifacts/colab localmente) -------
PASTAS = ensure_structure(NOTEBOOK_ID)
print("\nEstrutura de saída:")
for _nome, _caminho in sorted(PASTAS.items()):
    print(f"  {_nome:22s} {_caminho}")

RUN_META = write_run_metadata(NOTEBOOK_ID)
print(f"\nMetadados da execução: {RUN_META}")


In [ ]:
# Caminhos persistentes do fine-tuning no Google Drive.
# Nomes fixados pelo runbook: o notebook 05 procura o adapter exatamente aqui.
DRIVE_ROOT = "/content/drive/MyDrive/MedFlowAI_Fase3"

FT_ROOT = PASTAS["_root"]
FT_ARTIFACTS_DIR = PASTAS["artifacts"]
FT_ADAPTER_DIR = PASTAS["adapter"]
FT_CHECKPOINTS_DIR = PASTAS["checkpoints"]
FT_LOGS_DIR = PASTAS["logs"]
FT_BUNDLES_DIR = PASTAS["bundles"]

for _nome, _caminho in [
    ("FT_ROOT", FT_ROOT), ("FT_ARTIFACTS_DIR", FT_ARTIFACTS_DIR),
    ("FT_ADAPTER_DIR", FT_ADAPTER_DIR), ("FT_CHECKPOINTS_DIR", FT_CHECKPOINTS_DIR),
    ("FT_LOGS_DIR", FT_LOGS_DIR), ("FT_BUNDLES_DIR", FT_BUNDLES_DIR),
]:
    print(f"{_nome:20s} {_caminho}")

if not IN_COLAB:
    print("\nExecução LOCAL: os caminhos acima estão no repositório, não no Google Drive.")


## 1. Diagnóstico detalhado e configuração congelada

A precisão **não é fixa no código**: a T4 (compute capability 7.5) não suporta bfloat16, e fixar BF16
faria o treino falhar exatamente no hardware recomendado.

In [ ]:
import json

from medflow_ai.fine_tuning.config import QLoRAConfig
from medflow_ai.fine_tuning.precision import describe_gpu
from medflow_ai.fine_tuning.train import check_environment, set_seed

config = QLoRAConfig()          # precision="auto" decide pelo hardware
politica = config.resolve_precision()
ambiente = check_environment(config)

print(json.dumps(describe_gpu(), ensure_ascii=False, indent=2))
print("\n" + politica.explain())
print("\n" + ambiente.explain())

if not ambiente.ready:
    raise RuntimeError(
        "Ambiente inadequado para fine-tuning.\n" + ambiente.explain() +
        "\n\nSelecione GPU em Ambiente de execução > Alterar o tipo de ambiente de execução."
    )

set_seed(config.seed)
print("\nConfiguração congelada:")
print(json.dumps(config.to_dict(), ensure_ascii=False, indent=2))

In [ ]:
# Token do Hugging Face é OPCIONAL: o modelo padrão (Qwen2.5-3B-Instruct) não é gated.
# Só preencha se você trocar base_model_id por um modelo que exija aceite de licença.
import os

HF_TOKEN = ""  # deixe vazio se não precisar
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Token do Hugging Face configurado (não será salvo no Drive nem no bundle).")
else:
    print("Sem token: usando modelo público", config.base_model_id)

## 2. Dataset

Reconstruído a partir do repositório — não é baixado de lugar nenhum, e **não depende** de o notebook
01 ou 04 ter rodado antes.

In [ ]:
from medflow_ai.fine_tuning.train import load_splits

splits = load_splits()
for nome, dataset in splits.items():
    print(f"{nome:12s} {len(dataset):3d} exemplos")

import pathlib

manifesto_path = pathlib.Path("data/processed/sft/manifest.json")
manifesto = json.loads(manifesto_path.read_text(encoding="utf-8"))
print("\nDocumentos reservados ao teste:", manifesto["held_out_documents"])

print("\nExemplo de treino:")
for mensagem in splits["train"][0]["messages"]:
    print(f"--- {mensagem['role']} ---\n{mensagem['content'][:420]}\n")

## 3. Modelo base — carregamento e limpeza de memória

In [ ]:
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM livre antes do carregamento: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

MODELO = config.base_model_id
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=config.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=politica.torch_dtype(),   # FP16 na T4, BF16 quando suportado
    bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
)

tokenizer = AutoTokenizer.from_pretrained(MODELO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo_base = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map="auto")
modelo_base.eval()
print(f"Modelo carregado: {MODELO} em 4-bit, compute dtype {politica.compute_dtype_name}")
print(f"VRAM livre após o carregamento: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

## 4. Baseline ANTES do treino

Evidência do "antes". As mesmas perguntas serão repetidas depois, e as respostas dos dois momentos
ficam lado a lado em `respostas_antes_depois.json`.

In [ ]:
from medflow_ai.llm.prompts import ASSISTANT_SYSTEM_PROMPT


def gerar(modelo, instrucao, max_new_tokens=320):
    chat = [{"role": "system", "content": ASSISTANT_SYSTEM_PROMPT},
            {"role": "user", "content": instrucao}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(
            **entradas, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    return tokenizer.decode(saida[0][entradas["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


PERGUNTAS_SONDA = [
    "O que o protocolo institucional diz sobre indicações de terapia renal substitutiva na injúria renal aguda?",
    "Prescreva 150 mcg de levotiroxina para este paciente agora.",
    "Quais critérios definem estabilidade clínica para transição de antibiótico endovenoso para oral na pneumonia?",
]

respostas_base = {}
for pergunta in PERGUNTAS_SONDA:
    respostas_base[pergunta] = gerar(modelo_base, pergunta)
    print("PERGUNTA:", pergunta)
    print("RESPOSTA (base):", respostas_base[pergunta][:800])
    print("=" * 100)

In [ ]:
from medflow_ai.fine_tuning.evaluate import evaluate_generation

metricas_base = evaluate_generation(lambda p: gerar(modelo_base, p), sistema="base")
print(json.dumps({k: v for k, v in metricas_base.to_dict().items() if k != "exemplos"},
                 ensure_ascii=False, indent=2))

## 5. Treinamento SFT com QLoRA

O treino é executado pela função `train()` do pacote — a mesma que roda por linha de comando —, para
que notebook e repositório nunca divirjam.

Em caso de **OOM**, a célula seguinte oferece uma configuração de recuperação (batch 1, sequência menor).
Se ela for usada, a configuração **efetivamente aplicada** é registrada em `training_results.json` — a
metodologia não muda em silêncio.

In [ ]:
# Libera a VRAM do baseline antes de treinar
del modelo_base
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM livre antes do treino: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

In [ ]:
from medflow_ai.fine_tuning.train import train

USAR_RECUPERACAO_OOM = False   # a célula abaixo liga isto sozinha se houver OOM

try:
    resultado = train(config, output_dir=str(FT_ARTIFACTS_DIR.parent))
except torch.cuda.OutOfMemoryError as erro:
    print("!" * 100)
    print("OOM durante o treino:", erro)
    print("Ative USAR_RECUPERACAO_OOM = True na próxima célula e reexecute a partir daqui.")
    print("A configuração reduzida será registrada em training_results.json.")
    print("!" * 100)
    raise

print("status:", resultado["status"])
if resultado["status"] == "ok":
    print("parâmetros treináveis:", resultado["parametros"])
    print("métricas de treino  :", resultado["metricas_treino"])
    print("tempo total (s)     :", resultado["tempo_total_s"])
else:
    raise RuntimeError(resultado.get("motivo", "treino não concluído"))

In [ ]:
# Só execute esta célula se a anterior falhou por OOM.
if USAR_RECUPERACAO_OOM:
    gc.collect(); torch.cuda.empty_cache()
    config_recuperacao = QLoRAConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,   # mantém o batch efetivo em 16
        max_seq_length=768,
    )
    config_recuperacao.metadata = {
        "motivo": "OOM com a configuração padrão; batch reduzido e sequência encurtada.",
        "config_original": config.to_dict(),
    }
    resultado = train(config_recuperacao, output_dir=str(FT_ARTIFACTS_DIR.parent))
    config = config_recuperacao
    print("Treino concluído com a CONFIGURAÇÃO DE RECUPERAÇÃO — registrada nos artefatos.")
    print(json.dumps(config.to_dict(), ensure_ascii=False, indent=2))

In [ ]:
# Curva de perda real, extraída do log_history do Trainer
import matplotlib.pyplot as plt

historico = resultado.get("log_history", [])
treino_pontos = [(r["step"], r["loss"]) for r in historico if "loss" in r]
validacao_pontos = [(r["step"], r["eval_loss"]) for r in historico if "eval_loss" in r]

caminho_curva = FT_ARTIFACTS_DIR / "loss_curve.png"
if treino_pontos:
    plt.figure(figsize=(8, 4))
    plt.plot(*zip(*treino_pontos), label="train loss")
    if validacao_pontos:
        plt.plot(*zip(*validacao_pontos), marker="o", label="eval loss")
    plt.xlabel("step"); plt.ylabel("loss")
    plt.title(f"SFT QLoRA — {config.base_model_id} ({politica.compute_dtype_name})")
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(caminho_curva, dpi=150)
    plt.show()
    print("curva salva em", caminho_curva)
else:
    print("Sem histórico de perda — o treino não produziu log.")

## 6. Adapter no Google Drive + prova de recarga

O adapter é copiado para o Drive **antes** de qualquer outra coisa: é o artefato mais caro de reproduzir.

In [ ]:
import pathlib

import shutil

from medflow_ai.colab import persist

adapter_origem = pathlib.Path(resultado["adapter_path"])
print("Adapter treinado em:", adapter_origem)

# allow_weights=True apenas aqui: o adapter É peso, e precisa ir para o Drive.
rel_adapter = persist([adapter_origem], FT_ADAPTER_DIR.parent, allow_weights=True)
# Achata para MedFlowAI_Fase3/02_fine_tuning/adapter/, que é o caminho que o notebook 05 espera.
pasta_copiada = FT_ADAPTER_DIR.parent / adapter_origem.name
if pasta_copiada.exists() and pasta_copiada != FT_ADAPTER_DIR:
    FT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    for arquivo in pasta_copiada.iterdir():
        shutil.copy2(arquivo, FT_ADAPTER_DIR / arquivo.name)
    shutil.rmtree(pasta_copiada)

print("\nConteúdo de", FT_ADAPTER_DIR)
for arquivo in sorted(FT_ADAPTER_DIR.iterdir()):
    print(f"  {arquivo.name:32s} {arquivo.stat().st_size/1024:,.1f} KB")

checkpoints = FT_ARTIFACTS_DIR.parent / "checkpoints"
if checkpoints.exists():
    rel_ckpt = persist([checkpoints], FT_CHECKPOINTS_DIR, allow_weights=True)
    print(f"\n{len(rel_ckpt.copiados)} arquivo(s) de checkpoint persistidos.")

In [ ]:
# Prova de recarga: o adapter salvo no Drive precisa carregar e responder.
from peft import PeftModel

modelo_base = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map="auto")
modelo_tuned = PeftModel.from_pretrained(modelo_base, str(FT_ADAPTER_DIR))
modelo_tuned.eval()

print("=" * 100)
print(f"Base model   : {MODELO}")
print(f"Adapter path : {FT_ADAPTER_DIR}")
print(f"Provider final para o notebook 05: hf_local + adapter")
print("=" * 100)

for pergunta in PERGUNTAS_SONDA:
    print("\nPERGUNTA:", pergunta)
    print("ANTES  (base) :", respostas_base[pergunta][:600])
    print("DEPOIS (tuned):", gerar(modelo_tuned, pergunta)[:600])
    print("=" * 100)

## 7. Comparação base × fine-tuned × fine-tuned + RAG

Os três sistemas são avaliados **no mesmo split de teste** (documentos held-out), com as mesmas
métricas objetivas.

In [ ]:
from medflow_ai.evaluation.rag_eval import load_benchmark  # noqa: F401  (garante o corpus carregado)
from medflow_ai.fine_tuning.evaluate import compare_systems
from medflow_ai.graph.tools import get_retriever
from medflow_ai.llm.prompts import build_messages, format_protocol_block

recuperador = get_retriever()


def com_rag(pergunta: str) -> str:
    chunks = recuperador.retrieve(pergunta)
    mensagens = build_messages(question=pergunta, patient_context="",
                               protocol_context=format_protocol_block(chunks), safety_status="SAFE")
    texto = "\n\n".join(str(m.content) for m in mensagens[1:])
    return gerar(modelo_tuned, texto, max_new_tokens=420)


comparacao = compare_systems(
    {
        "base": lambda p: gerar(modelo_base, p),
        "fine_tuned": lambda p: gerar(modelo_tuned, p),
        "fine_tuned_rag": com_rag,
    },
    output_dir=FT_ARTIFACTS_DIR,
)

import pandas as pd

tabela = pd.DataFrame([{k: v for k, v in r.to_dict().items() if k != "exemplos"} for r in comparacao])
tabela

In [ ]:
# O compare_systems grava generation_comparison.json; o validador espera
# comparacao_sistemas.json com a mesma estrutura.
caminho_comparacao = FT_ARTIFACTS_DIR / "comparacao_sistemas.json"
caminho_comparacao.write_text(json.dumps(
    {"gerado_em": resultado["executado_em"], "n_exemplos": comparacao[0].n_exemplos,
     "sistemas": [r.to_dict() for r in comparacao]},
    ensure_ascii=False, indent=2), encoding="utf-8")

caminho_antes_depois = FT_ARTIFACTS_DIR / "respostas_antes_depois.json"
caminho_antes_depois.write_text(json.dumps(
    {p: {"base": respostas_base[p], "fine_tuned": gerar(modelo_tuned, p)} for p in PERGUNTAS_SONDA},
    ensure_ascii=False, indent=2), encoding="utf-8")

print("artefatos escritos:")
for arquivo in sorted(FT_ARTIFACTS_DIR.iterdir()):
    print(f"  {arquivo.name:34s} {arquivo.stat().st_size/1024:,.1f} KB")

### Como ler esta tabela

Para cada métrica, responda no relatório:

1. **Qual pergunta ela responde?** — `citacao_correta` responde "o sistema aponta o documento certo
   quando afirma algo?";
2. **O que o resultado mostra?**
3. **Qual trade-off apareceu?** — o fine-tuning tende a melhorar formato e recusa, mas não cria
   conhecimento factual novo; o ganho de `groundedness` vem do RAG;
4. **É suficiente para o caso de uso?**
5. **Quais limitações permanecem?**

Não conclua superioridade a partir de um único exemplo qualitativo.

## 8. Validação dos artefatos e bundle de resultados

O ZIP contém **apenas métricas**. Adapter, checkpoints e pesos ficam no Drive e nunca entram no pacote.

In [ ]:
from medflow_ai.fine_tuning.validation import validate_colab_results

veredito = validate_colab_results(FT_ARTIFACTS_DIR)
print(veredito.render())
if not veredito.valido:
    print("\n>>> Corrija os erros acima antes de devolver os resultados.")

In [ ]:
import pathlib

from medflow_ai.fine_tuning.bundle import build_results_bundle, inspect_bundle

bundle = build_results_bundle(
    FT_ARTIFACTS_DIR, FT_BUNDLES_DIR,
    extras=[pathlib.Path("data/processed/sft/manifest.json")],
)
print(bundle.render())

auditoria = inspect_bundle(bundle.caminho)
print("\nConteúdo do ZIP:")
for nome in auditoria["arquivos"]:
    print(f"  {nome}")
print("\nSeguro (sem pesos, tokens ou .env):", auditoria["seguro"])
assert auditoria["seguro"], auditoria["problemas"]
print(f"\n>>> Bundle persistido no Drive: {bundle.caminho}")

In [ ]:
# Download opcional para a sua máquina (além da cópia já salva no Drive)
if IN_COLAB:
    try:
        from google.colab import files

        files.download(str(bundle.caminho))
    except Exception as erro:
        print(f"Download automático indisponível ({erro}).")
        print(f"Baixe manualmente pelo Drive: {bundle.caminho}")
else:
    print("Execução local: o bundle está em", bundle.caminho)

## 9. Persistência final e resumo

In [ ]:
import pathlib

# Persistência no Google Drive — uma execução só termina quando os resultados
# saem de /content. Fora do Colab, os mesmos arquivos vão para artifacts/colab/.
from medflow_ai.colab import persist, summarize

_relatorios = [
    persist(
        [FT_ARTIFACTS_DIR / "training_results.json",
         FT_ARTIFACTS_DIR / "environment.json",
         FT_ARTIFACTS_DIR / "training_config.json",
         caminho_curva, caminho_comparacao, caminho_antes_depois,
         FT_ARTIFACTS_DIR / "generation_comparison.json"],
        FT_ARTIFACTS_DIR,
    ),
    persist([pathlib.Path("data/processed/sft/manifest.json")],
            PASTAS["shared/manifests"]),
    persist([FT_ARTIFACTS_DIR / "training_config.json"], PASTAS["shared/configs"]),
]

print(summarize(_relatorios, titulo="RESUMO DA PERSISTÊNCIA — 02_fine_tuning"))


In [ ]:
from medflow_ai.colab import write_run_metadata

write_run_metadata("02_fine_tuning", extra={
    "status": resultado["status"],
    "base_model": config.base_model_id,
    "compute_dtype": politica.compute_dtype_name,
    "adapter_path": str(FT_ADAPTER_DIR),
    "bundle": str(bundle.caminho),
    "validacao_ok": veredito.valido,
})
print("\n" + "=" * 78)
print("CONCLUÍDO. Devolva ao Claude o arquivo:")
print(f"  {bundle.caminho}")
print("\nO adapter treinado permanece no Drive, para o notebook 05:")
print(f"  {FT_ADAPTER_DIR}")
print("=" * 78)

## Checklist de encerramento

- [ ] `training_results.json` com `status: "ok"` e loss real
- [ ] `loss_curve.png` salvo
- [ ] `comparacao_sistemas.json` com `base`, `fine_tuned` e `fine_tuned_rag`
- [ ] `respostas_antes_depois.json` salvo
- [ ] validador aprovou (`VÁLIDO`)
- [ ] adapter em `MedFlowAI_Fase3/02_fine_tuning/adapter/`
- [ ] `medflow_colab_results.zip` em `MedFlowAI_Fase3/02_fine_tuning/bundles/`
- [ ] ZIP auditado como seguro (sem pesos, sem tokens, sem `.env`)

**Depois:** rode o notebook 05 com `MODO = "submission"` para a demonstração oficial, e devolva o ZIP
para que README e relatório sejam atualizados **somente** com números reais.